In [1]:
import os
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

PROJECT_ROOT = os.path.dirname(os.getcwd())
TARGET_PATH = os.path.join(PROJECT_ROOT, "data", "tables", "bronze_stream")

builder = (
    SparkSession.builder.appName("Query_Bronze")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

df = spark.read.format("delta").load(TARGET_PATH)
latest_ingested_records = df.orderBy(F.col("_ingested_at").desc())

latest_ingested_records.select("kafka_key", "value", "partition", "offset", "_ingested_at").show(5, truncate=False)

:: loading settings :: url = jar:file:/Users/intuitivecode/Code/deltastreams/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/intuitivecode/.ivy2/cache
The jars for the packages stored in: /Users/intuitivecode/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6c537174-bfa8-4d98-bed1-19a0649bd50c;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 89ms :: artifacts dl 3ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default    

+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+-----------------------+
|kafka_key|value                                                                                                                                                                                                                                                                               |partition|offset|_ingested_at           |
+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------+------+-----------------------+
|NULL     

In [3]:
# Load Delta Table Reference and Record Count
df = spark.read.format("delta").load(TARGET_PATH)
delta_table = DeltaTable.forPath(spark, TARGET_PATH)

print(f"Total Bronze Records Ingested: {df.count():,}")

Total Bronze Records Ingested: 62,453
